In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import re

# precompile for speed
_JSON_OVERALL = re.compile(r'(?i)"overall_status"\s*:\s*".*?"\s*,?')
_JSON_WHY     = re.compile(r'(?i)"why_stopped"\s*:\s*".*?"\s*,?')
_XML_OVERALL  = re.compile(r'(?is)<\s*overall_status\s*>.*?<\s*/\s*overall_status\s*>')
_XML_WHY      = re.compile(r'(?is)<\s*why_stopped\s*>.*?<\s*/\s*why_stopped\s*>')

# success/fail stems (common inflections)
_SUCCESS_STEMS = re.compile(
    r'(?i)\b(?:success|successful|successfully|succeed|succeeds|succeeded|succeeding)\b'
)
_FAIL_STEMS = re.compile(
    r'(?i)\b(?:fail|fails|failed|failing|failure|failures)\b'
)

# explicit outcome phrases (safer than nuking every "positive"/"negative")
_POS_NEG_OUTCOME = re.compile(
    r'(?i)\b(positive|negative)\b(?:\W+\w+){0,3}?\b(outcome|result|trial|study|response|endpoint)s?\b'
)

# legacy human-readable lines (if they ever appear)
_OVERALL_LINE = re.compile(r'(?i)\boverall\s+status\s*:\s*.*?(?:\n|$)')
_WHY_LINE     = re.compile(r'(?i)\bwhy\s+stop(?:ped)?\s*:\s*.*?(?:\n|$)')

def remove_leakage(text: str) -> str:
    # strip structured fields first
    text = _JSON_OVERALL.sub(' ', text)
    text = _JSON_WHY.sub(' ', text)
    text = _XML_OVERALL.sub(' ', text)
    text = _XML_WHY.sub(' ', text)
    text = _OVERALL_LINE.sub(' ', text)
    text = _WHY_LINE.sub(' ', text)

    # explicit outcome phrasing with positive/negative
    text = _POS_NEG_OUTCOME.sub(' ', text)

    # success/fail stems (broad but focused on outcome words)
    text = _SUCCESS_STEMS.sub(' ', text)
    text = _FAIL_STEMS.sub(' ', text)

    # tidy whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [3]:
import os
from typing import List, Tuple, Dict, Optional

def _find_labels_file(dir_path: str) -> str:
    """
    Return the path to the labels file inside dir_path.
    Accepts either 'labels.txt' or 'label.txt'. Raises if neither exists.
    """
    candidates = ["labels.txt", "label.txt"]
    for fname in candidates:
        fp = os.path.join(dir_path, fname)
        if os.path.isfile(fp):
            return fp
    raise FileNotFoundError(
        f"No labels file found in {dir_path}. Expected one of: {', '.join(candidates)}"
    )

def _load_labels(labels_path: str, expected_n: int = 400) -> List[int]:
    """
    Load labels (0/1) from labels_path, ignoring empty lines.
    Validates count and values.
    """
    with open(labels_path, "r", encoding="utf-8") as f:
        raw = f.readlines()

    labels: List[int] = []
    for line in raw:
        s = line.strip()
        if not s:
            continue
        if s not in {"0", "1"}:
            raise ValueError(f"Invalid label '{s}' in {labels_path}. Only '0' or '1' are allowed.")
        labels.append(int(s))

    if len(labels) != expected_n:
        raise ValueError(
            f"Found {len(labels)} labels in {labels_path}, but expected {expected_n}."
        )
    return labels

def load_synthetic_group(dir_path: str, expected_n: int = 400) -> List[Tuple[str, int]]:
    """
    Load one synthetic group from `dir_path`.
    - Reads labels from labels.txt/label.txt (400 lines of 0/1, empties ignored)
    - Reads synthetic_000000.txt ... synthetic_000399.txt
    - Applies remove_leakage(text) to each text
    Returns: list of (clean_text, label) with length `expected_n`, index-aligned by filename order.
    """
    if not os.path.isdir(dir_path):
        raise NotADirectoryError(f"Not a directory: {dir_path}")

    labels_path = _find_labels_file(dir_path)
    labels = _load_labels(labels_path, expected_n=expected_n)

    data: List[Tuple[str, int]] = []
    for i in range(expected_n):
        fname = f"synthetic_{i:06d}.txt"
        fpath = os.path.join(dir_path, fname)
        if not os.path.isfile(fpath):
            raise FileNotFoundError(f"Missing file: {fpath}")
        with open(fpath, "r", encoding="utf-8") as f:
            text = f.read()
        clean = remove_leakage(text)  # assumes your remove_leakage is defined/imported
        data.append((clean, labels[i]))

    return data

def load_all_synthetic_groups() -> Dict[str, List[Tuple[str, int]]]:
    """
    Loads all seven groups you specified, keyed by short group name.
    """
    groups = {
        "no_reasoning": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/ablations/no_reasoning",
        "no_retrieval": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/ablations/no_retrieval",
        "baseline": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/baseline",
        "positive_negative_same_as_is": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/prompt_variants/positive_negative_same_as_is",
        "success_failure_mixed_head_as_is": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/prompt_variants/success_failure_mixed_head_as_is",
        "success_failure_mixed_tail_as_is": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/prompt_variants/success_failure_mixed_tail_as_is",
        "success_failure_same_shuffled": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/prompt_variants/success_failure_same_shuffled",
    }
    loaded: Dict[str, List[Tuple[str, int]]] = {}
    for name, path in groups.items():
        loaded[name] = load_synthetic_group(path, expected_n=400)
    return loaded


In [4]:
import os
import re
import json
import random
import xml.etree.ElementTree as ET
from typing import List, Tuple, Optional
import pandas as pd

# ---- XML helpers ----
def element_to_dict(el):
    children = list(el)
    if not children:
        return el.text
    result = {}
    for child in children:
        child_dict = element_to_dict(child)
        if child.tag in result:
            if not isinstance(result[child.tag], list):
                result[child.tag] = [result[child.tag]]
            result[child.tag].append(child_dict)
        else:
            result[child.tag] = child_dict
    return result

def xml_to_dict(element):
    return {element.tag: element_to_dict(element)}

def read_xml_file(file_path: str) -> dict:
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        return xml_to_dict(root)
    except Exception:
        return {}

def _xml_path_for_study(trials_dir: str, study_id: str) -> Optional[str]:
    """
    Builds the expected XML path:
      <trials_dir>/<study_id[:7]>xxxx/<study_id>.xml
    Falls back to <trials_dir>/<study_id>.xml if subfolder is absent.
    Returns a path if it exists, else None.
    """
    subdir = os.path.join(trials_dir, f"{study_id[:7]}xxxx")
    p1 = os.path.join(subdir, f"{study_id}.xml")
    if os.path.isfile(p1):
        return p1
    p2 = os.path.join(trials_dir, f"{study_id}.xml")
    if os.path.isfile(p2):
        return p2
    return None

# ---- Main loader ----
def load_real_samples(
    trials_dir: str = "/content/drive/MyDrive/ColabRepos/REPO/data/trials",
    labels_csv_path: str = "/content/drive/MyDrive/ColabRepos/REPO/data/IQVIA/filtered_trial_outcomes_with_labels.csv",
    n_samples: int = 2100,
    seed: int = 42,
) -> List[Tuple[str, int]]:
    """
    Load `n_samples` real trial texts + labels at random (reproducible with `seed`).
    - trials_dir: folder containing trial XMLs (possibly under <prefix>xxxx subfolders)
    - labels_csv_path: CSV with at least columns ['studyid', 'label'] (0/1 or str)
    - n_samples: number of examples to return
    - seed: random seed for reproducibility

    Returns: list of (clean_text, label_int)
    """
    if not os.path.isdir(trials_dir):
        raise NotADirectoryError(f"Not a directory: {trials_dir}")
    if not os.path.isfile(labels_csv_path):
        raise FileNotFoundError(f"Labels CSV not found: {labels_csv_path}")

    # Load labels CSV
    df = pd.read_csv(labels_csv_path)
    if not {"studyid", "label"}.issubset(df.columns):
        raise ValueError("labels_csv must contain columns: 'studyid', 'label'.")

    # Normalize labels to 0/1
    def _to_int_label(x):
        s = str(x).strip().lower()
        if s in {"1", "success", "successful", "completed", "positive"}:
            return 1
        if s in {"0", "failure", "failed", "terminated", "negative"}:
            return 0
        # fallback: try cast
        return int(x)

    df["label"] = df["label"].apply(_to_int_label)

    # Shuffle rows with seed
    rng = random.Random(seed)
    indices = list(range(len(df)))
    rng.shuffle(indices)

    out: List[Tuple[str, int]] = []
    tried = 0

    for idx in indices:
        row = df.iloc[idx]
        study_id = str(row["studyid"]).strip()
        label = int(row["label"])

        xml_path = _xml_path_for_study(trials_dir, study_id)
        if not xml_path:
            tried += 1
            continue

        xml_dict = read_xml_file(xml_path)
        if not xml_dict:
            tried += 1
            continue

        # Create a compact string representation
        xml_string = json.dumps(xml_dict, ensure_ascii=False)

        # Call your leakage remover
        clean_text = remove_leakage(xml_string)

        out.append((clean_text, label))

        if len(out) >= n_samples:
            break

    if len(out) < n_samples:
        raise RuntimeError(
            f"Could only load {len(out)} samples (requested {n_samples}). "
            f"Checked {tried} items; many may be missing/unreadable."
        )

    return out


In [5]:
# === Hybrid training experiment: 20% synthetic (400) + 80% real (1600) ===
# For 7 synthetic groups, seeds 42/43/44, BioBERT fine-tuning + metrics
# Requirements: transformers, torch, sklearn, pandas, numpy

import os, random, numpy as np, torch
from typing import List, Tuple, Dict
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------
GROUP_DIRS = {
    "no_reasoning": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/ablations/no_reasoning",
    "no_retrieval": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/ablations/no_retrieval",
    "baseline": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/baseline",
    "positive_negative_same_as_is": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/prompt_variants/positive_negative_same_as_is",
    "success_failure_mixed_head_as_is": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/prompt_variants/success_failure_mixed_head_as_is",
    "success_failure_mixed_tail_as_is": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/prompt_variants/success_failure_mixed_tail_as_is",
    "success_failure_same_shuffled": "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/prompt_variants/success_failure_same_shuffled",
}

TRIALS_DIR = "/content/drive/MyDrive/ColabRepos/REPO/data/trials"
LABELS_CSV = "/content/drive/MyDrive/ColabRepos/REPO/data/IQVIA/filtered_trial_outcomes_with_labels.csv"

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
LR = 1e-5
BATCH_SIZE = 8
EPOCHS = 7
MAX_LEN = 512

SEEDS = [42, 43, 44]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------
# Helpers (tokenize / dataset / training / evaluation)
# ---------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_pairs(pairs: List[Tuple[str, int]]):
    texts = [t for (t, _) in pairs]
    labels = [y for (_, y) in pairs]
    enc = tokenizer(
        texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"
    )
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

def to_loader(encodings, batch_size=BATCH_SIZE, shuffle=False, seed=None):
    ds = TensorDataset(encodings["input_ids"], encodings["attention_mask"], encodings["labels"])
    if seed is not None:
        g = torch.Generator()
        g.manual_seed(seed)
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=g)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def safe_auc(y_true, y_prob):
    """Return (roc_auc, pr_auc) with graceful fallback if y_true has a single class."""
    try:
        roc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc = float("nan")
    try:
        pr = average_precision_score(y_true, y_prob)
    except Exception:
        pr = float("nan")
    return roc, pr

def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids, attn, labels = [x.to(DEVICE) for x in batch]
            out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
            logits = out.logits
            all_logits.append(logits.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
    all_logits = np.vstack(all_logits)
    y_true = np.concatenate(all_labels)
    y_pred = all_logits.argmax(axis=1)
    # softmax proba for positive class
    exp = np.exp(all_logits - all_logits.max(axis=1, keepdims=True))
    probs = exp / exp.sum(axis=1, keepdims=True)
    y_prob_pos = probs[:, 1]
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    roc, pr = safe_auc(y_true, y_prob_pos)
    return acc, prec, rec, roc, pr

def train_one_run(train_loader, val_loader=None, seed=0):
    set_all_seeds(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    model.train()
    for _ in range(EPOCHS):
        for batch in train_loader:
            input_ids, attn, labels = [x.to(DEVICE) for x in batch]
            out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
            loss = out.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model

# ---------------------------------------------------------------------
# Core experiment function
# ---------------------------------------------------------------------
def run_hybrid_for_group(group_name: str, group_dir: str) -> Dict[str, Dict[str, float]]:
    """
    For a given synthetic group:
      - For seeds 40/41/42:
        - Load 2100 real samples with that seed
        - Split real: 1600 train + 250 val + 250 test
        - Load 400 synthetic samples from group_dir
        - Make train = 1600 real + 400 synthetic
        - Fine-tune and evaluate on real test (250)
      - Return mean & variance across seeds for each metric
    """
    # Load all 400 synthetic samples (already leakage-cleaned)
    synth_pairs = load_synthetic_group(group_dir, expected_n=400)

    # Storage for 3 runs
    accs, precs, recs, rocs, prs = [], [], [], [], []

    for seed in SEEDS:
        # 1) Real set for this seed
        real_2100 = load_real_samples(
            trials_dir=TRIALS_DIR,
            labels_csv_path=LABELS_CSV,
            n_samples=2100,
            seed=seed
        )

        # 2) Split: 1600 train + 500 leftover
        set_all_seeds(seed)
        idx = list(range(len(real_2100)))  # 0..2099
        random.shuffle(idx)
        real_train_idx = idx[:1600]
        leftover_idx = idx[1600:]  # 500 items
        # 3) Split leftover into 250 val / 250 test
        random.shuffle(leftover_idx)
        real_val_idx = leftover_idx[:250]
        real_test_idx = leftover_idx[250:500]

        real_train_pairs = [real_2100[i] for i in real_train_idx]
        real_val_pairs   = [real_2100[i] for i in real_val_idx]
        real_test_pairs  = [real_2100[i] for i in real_test_idx]

        # 4) Build train set: 1600 real + 400 synthetic = 2000
        train_pairs = real_train_pairs + synth_pairs
        # Shuffle training set (seeded)
        set_all_seeds(seed)
        random.shuffle(train_pairs)

        # 5) Tokenize
        train_enc = tokenize_pairs(train_pairs)
        val_enc   = tokenize_pairs(real_val_pairs)
        test_enc  = tokenize_pairs(real_test_pairs)

        # 6) DataLoaders
        train_loader = to_loader(train_enc, batch_size=BATCH_SIZE, shuffle=True,  seed=seed)
        val_loader   = to_loader(val_enc,   batch_size=BATCH_SIZE, shuffle=False, seed=seed)
        test_loader  = to_loader(test_enc,  batch_size=BATCH_SIZE, shuffle=False, seed=seed)

        # 7) Train and evaluate
        model = train_one_run(train_loader, val_loader=val_loader, seed=seed)
        acc, prec, rec, roc, pr = evaluate(model, test_loader)

        accs.append(acc); precs.append(prec); recs.append(rec); rocs.append(roc); prs.append(pr)

        # cleanup
        del model
        torch.cuda.empty_cache()

    # Aggregate: mean & variance (population variance)
    def agg(x):
        x = np.array(x, dtype=float)
        return float(np.nanmean(x)), float(np.nanvar(x))

    results = {
        "Accuracy":  {"mean": agg(accs)[0], "variance": agg(accs)[1]},
        "Precision": {"mean": agg(precs)[0], "variance": agg(precs)[1]},
        "Recall":    {"mean": agg(recs)[0], "variance": agg(recs)[1]},
        "ROC-AUC":   {"mean": agg(rocs)[0], "variance": agg(rocs)[1]},
        "PR-AUC":    {"mean": agg(prs)[0],  "variance": agg(prs)[1]},
    }
    return results


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
# ---------------------------------------------------------------------
# Run all 7 groups and print results
# ---------------------------------------------------------------------
all_results = {}
for name, path in GROUP_DIRS.items():
    print(f"\n=== Running group: {name} ===")
    res = run_hybrid_for_group(name, path)
    all_results[name] = res
    for metric, vals in res.items():
        print(f"{metric:10s}  mean={vals['mean']:.4f}  variance={vals['variance']:.6f}")


=== Running group: no_reasoning ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Accuracy    mean=0.6533  variance=0.000558
Precision   mean=0.6906  variance=0.000175
Recall      mean=0.8112  variance=0.007949
ROC-AUC     mean=0.6857  variance=0.000957
PR-AUC      mean=0.7795  variance=0.000144

=== Running group: no_retrieval ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Accuracy    mean=0.6320  variance=0.001163
Precision   mean=0.7366  variance=0.000086
Recall      mean=0.6422  variance=0.008396
ROC-AUC     mean=0.6781  variance=0.000530
PR-AUC      mean=0.7747  variance=0.000064

=== Running group: baseline ===


...

Accuracy    mean=0.6427  variance=0.000356
Precision   mean=0.6885  variance=0.000164
Recall      mean=0.7825  variance=0.001134
ROC-AUC     mean=0.6701  variance=0.001072
PR-AUC      mean=0.7624  variance=0.000509

=== Running group: success_failure_mixed_tail_as_is ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Accuracy    mean=0.6373  variance=0.000622
Precision   mean=0.7190  variance=0.000092
Recall      mean=0.6888  variance=0.002099
ROC-AUC     mean=0.6798  variance=0.000122
PR-AUC      mean=0.7714  variance=0.000107

=== Running group: success_failure_same_shuffled ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Accuracy    mean=0.6520  variance=0.000224
Precision   mean=0.7059  variance=0.000339
Recall      mean=0.7608  variance=0.003352
ROC-AUC     mean=0.6790  variance=0.000107
PR-AUC      mean=0.7660  variance=0.000101


In [7]:

# Optional: save results to JSON
try:
    import json
    out_path = "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/hybrid_20_80_results.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved results to: {out_path}")
except Exception as e:
    print(f"Could not save results: {e}")


Saved results to: /content/drive/MyDrive/ColabRepos/REPO/variants_ablations/hybrid_20_80_results.json
